# TICKET MJD-606 
De: Raquel Fernández · Jefa de ventas Cantabria
El ranking de mis comerciales de Cantabria, enero a septiembre de 2026 contra el mismo periodo de
2025, en un Excel que pueda abrir y reenviar. Con los números como los escribimos aquí, no como en in‐
glés.

ENTREGABLE
- entregables/mjd-606_ranking_cantabria.xlsx , y el mismo ranking en el notebook con formato español.

In [14]:
import os, sys
from pathlib import Path

RAIZ = Path.home() / "master-data-ia"
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [15]:
import duckdb
con = duckdb.connect("db/marejada.duckdb", read_only=True)
ranking = con.sql("""
SELECT c.comercial_asignado AS comercial,
sum(f.importe_neto) FILTER (WHERE d.anio = 2026) AS ventas_2026,
sum(f.importe_neto) FILTER (WHERE d.anio = 2025) AS ventas_2025
FROM gold.fact_ventas AS f
JOIN gold.dim_fecha AS d ON d.fecha_key = f.fecha_key
JOIN gold.dim_cliente AS c ON c.cliente_key = f.cliente_key
WHERE c.provincia = 'Cantabria' AND d.mes <= 9 AND d.anio IN (2025, 2026)
GROUP BY c.comercial_asignado
""").df()
con.close()
ranking["variacion_pct"] = (ranking["ventas_2026"] / ranking["ventas_2025"] - 1) * 100
ranking = ranking.sort_values("ventas_2026", ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1
ranking

,comercial,ventas_2026,ventas_2025,variacion_pct
1,Pablo Revuelta Gómez,1.818300e+06,1.642011e+06,10.736197
2,Raquel Fernández Cobo,1.708279e+06,1.662061e+06,2.780746
3,Daniel Gutiérrez Pérez,1.567632e+06,1.392262e+06,12.596014
4,Celia Gómez Sáiz,1.063105e+06,3.886159e+05,173.561933
5,Sara Ceballos Ortiz,1.045961e+06,8.477461e+05,23.381433
6,Lorena Arce Díaz,7.817259e+05,6.394953e+05,22.241080
7,Héctor Bustamante Lavín,NaN,5.964155e+05,NaN


Ahora, el formato español. Python escribe los números al estilo inglés ( 1,234.5 ). El truco habitual: formatear
en inglés y luego intercambiar la coma y el punto, pasando por un carácter intermedio:

In [16]:
from openpyxl.styles import Font
import pandas as pd

ruta = "entregables/mjd-606_ranking_cantabria.xlsx"
salida = ranking.rename(columns={
    "comercial": "Comercial", "ventas_2026": "Ventas ene-sep 2026 (€)",
    "ventas_2025": "Ventas ene-sep 2025 (€)", "variacion_pct": "Variación (%)",
})
salida["Variación (%)"] = salida["Variación (%)"] / 100

with pd.ExcelWriter(ruta, engine="openpyxl") as xls:
    salida.to_excel(xls, sheet_name="Ranking", index_label="Puesto")
    hoja = xls.sheets["Ranking"]
    for celda in hoja[1]:
        celda.font = Font(bold=True)
    for fila in hoja.iter_rows(min_row=2, min_col=3, max_col=4):
        for celda in fila:
            celda.number_format = "#,##0.00 €"
    for (celda,) in hoja.iter_rows(min_row=2, min_col=5, max_col=5):
        celda.number_format = "0.0%"
    for col, ancho in zip("ABCDE", [8, 26, 24, 24, 14]):
        hoja.column_dimensions[col].width = ancho

comprobacion = pd.read_excel(ruta)
print(comprobacion.dtypes)
round(comprobacion["Ventas ene-sep 2026 (€)"].sum(), 2) == round(ranking["ventas_2026"].sum(), 2)

Puesto                       int64
Comercial                      str
Ventas ene-sep 2026 (€)    float64
Ventas ene-sep 2025 (€)    float64
Variación (%)              float64
dtype: object


np.True_